In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import csv
import pandas as pd
from matplotlib.lines import Line2D
from scipy.integrate import quad
from functools import reduce
import operator
Cww = 1
Cgg = 0
cff=0
Cbb = 1
mz = 91.188
mw = 80.3692
cw = mw/mz
sw = np.sqrt(1 - cw**2) 
alpha = 1/128
alpha_s = 0.11 #mz
hbar_c = 1.973e-13  # GeV·cm

Caa = Cww + Cbb
Caz = cw**2 * Cww - sw**2 * Cbb
Czz = cw**4 * Cww + sw**4 * Cbb

In [ ]:
# 生成 MQ 在 1e-4–1e2 间的 500 个对数等距点
ma_vals = np.logspace(-5, 2, 500)  
Lambda_vals = 1
# 存成新的 DataFrame
df = pd.DataFrame({
    'MQ':      ma_vals,
    'gstar':   Lambda_vals
})
ma_grid = df['MQ'].values
Lambda_grid = df['gstar'].values

In [ ]:
#######################
##光子衰变函数
#######################
def Gamma_a_to_gg(ma_grid, Lambda_grid, fermions):
    Ceff = C_gamma_gamma_eff(ma_grid, fermions)
    return 4 * np.pi * alpha**2 * ma_grid**3 / Lambda_grid**2 * np.real(Ceff * np.conj(Ceff))
def C_gamma_gamma_eff(ma_grid, fermions):
    total = np.zeros_like(ma_grid, dtype=complex)
    for f in fermions:
        term = (
            f["Nc"] * f["Qf"]**2 * cff * B1_tau(f["mf"], ma_grid) / (16 * np.pi**2)
        )
        total += term
    return Caa + total + 2 * Cww * alpha / (np.pi * sw**2) * B2_tau(mw, ma_grid)
def c_qq(Qq, T3q, mq, ma_array):
    ln_mu2_mq2 = np.log(mz**2 / mq**2)
    ln_mu2_mZ2 = np.log(mz**2 / mz**2)
    ln_mu2_mW2 = np.log(mz**2 / mw**2)
    g_vals = g_tau(mq, ma_array)
    term1 = -12 * Qq**2 * alpha**2 * Caa * (ln_mu2_mq2 + g_vals)
    term2 = -12 * alpha**2 / (sw**2 * cw**2) * Caz * Qq * (T3q - 2 * Qq * sw**2) * (ln_mu2_mZ2 + 1.5)
    term3 = -12 * alpha**2 / (sw**4 * cw**4) * Czz * (Qq**2 * sw**4 - T3q * sw**2 * Qq + 1/8) * (ln_mu2_mZ2 + 0.5)
    term4 = -3 * alpha**2 /(sw**4) * (ln_mu2_mW2 + 0.5) * Cww
    return term1 + term2 + term3 + term4

def g_tau(ml, ma_array):
    g_vals = np.empty_like(ma_array, dtype=complex)
    small_mask = ml**2 < ma_array**2
    large_mask = ml**2 > ma_array**2
    # 小极限
    log_term = np.log(ma_array[small_mask]**2 / ml**2) - 1j * np.pi
    g_vals[small_mask] = -1/6 * log_term**2 + 2/3
    # 大极限
    g_vals[large_mask] = 7/3
    # 相等情况
    g_vals[~(small_mask | large_mask)] =  5 + 4/3*(-0.5 + 1j * np.pi/2)
    return g_vals

def build_sm_fermions(ma_grid):
    fermions = []
    sm_data = [
        # name, Qf, mf, T3, Nc
        ("u_L",  2/3, 2.2e-3,  +0.5, 3), ("u_R",  2/3, 2.2e-3,  0.0,  3),
        ("d_L", -1/3, 4.7e-3,  -0.5, 3), ("d_R", -1/3, 4.7e-3,  0.0,  3),
        ("c_L",  2/3, 1.28,    +0.5, 3), ("c_R",  2/3, 1.28,    0.0,  3),
        ("s_L", -1/3, 0.096,   -0.5, 3), ("s_R", -1/3, 0.096,   0.0,  3),
        ("t_L",  2/3, 173,     +0.5, 3), ("t_R",  2/3, 173,     0.0,  3),
        ("b_L", -1/3, 4.18,    -0.5, 3), ("b_R", -1/3, 4.18,    0.0,  3),
        ("e_L", -1.0, 0.511e-3,-0.5, 1), ("e_R", -1.0, 0.511e-3, 0.0, 1),
        ("mu_L",-1.0, 0.1057,  -0.5, 1), ("mu_R",-1.0, 0.1057,   0.0, 1),
        ("tau_L",-1.0, 1.777,  -0.5, 1), ("tau_R",-1.0, 1.777,   0.0, 1),
    ]
    for name, Qf, mf, T3, Nc in sm_data:
        cff = c_qq(Qf, T3, mf, ma_array=ma_grid)  # <-- 支持2D
        fermions.append({"name": name, "Nc": Nc, "Qf": Qf, "mf": mf, "cff": cff})
    return fermions
def B1_tau(ml, ma_grid):
    tau = 4 * ml**2 / ma_grid**2
    return 1 - tau * f_tau(ml, ma_grid)**2

def B2_tau(ml, ma_grid):
    tau = 4 * ml**2 / ma_grid**2
    return 1 - (tau - 1) * f_tau(ml, ma_grid)**2

def f_tau(ml, ma_grid):
    tau = 4 * ml**2 / ma_grid**2
    result = np.empty_like(tau, dtype=complex)
    mask1 = tau >= 1
    mask2 = tau < 1
    result[mask1] = np.arcsin(1 / np.sqrt(tau[mask1]))
    result[mask2] = (np.pi/2 + 0.5j * np.log((1 + np.sqrt(1 - tau[mask2])) / (1 - np.sqrt(1 - tau[mask2]))))
    return result

In [ ]:
#######################
##轻子衰变函数
#######################
def Gamma_a_to_ll(ma_grid, ml, cell_eff, Lambda_grid):
    Gamma = np.zeros_like(ma_grid)
    mask = ma_grid >= 2 * ml
    m = ma_grid[mask]
    L = Lambda_grid[mask]
    kin = np.sqrt(1 - 4 * ml**2 / m**2)
    pref = m * ml**2 / (8 * np.pi * L**2)
    Gamma[mask] = pref * np.abs(cell_eff[mask])**2 * kin
    return Gamma

def compute_lepton_decay_widths(ma_grid, Lambda_grid):
    results = {}
    lepton_data = {
        "e":  (0.511e-3, -0.5),
        "mu": (0.1057,   -0.5),
        "tau":(1.777,    -0.5),
    }
    for name, (ml, T3) in lepton_data.items():
        c_eff = c_qq(Qq=-1.0, T3q=T3, mq=ml, ma_array=ma_grid)  # ma_array must be 2D
        Gamma = Gamma_a_to_ll(ma_grid, ml=ml, cell_eff=c_eff, Lambda_grid=Lambda_grid)
        results[name] = Gamma
    return results

In [ ]:
#######################
##轻夸克和重夸克衰变
#######################
def compute_light_quark_decay_widths(ma_grid, Lambda_grid):
    alpha_s_vals = alpha_s * np.ones_like(ma_grid)
    results = {
        "uu": np.zeros_like(ma_grid, dtype=complex),
        "dd": np.zeros_like(ma_grid, dtype=complex),
        "ss": np.zeros_like(ma_grid, dtype=complex)
    }
    quark_groups = {
        "uu": [("u_L", 2/3, 2.2e-3, +0.5), ("u_R", 2/3, 2.2e-3, 0.0)],
        "dd": [("d_L", -1/3, 4.7e-3, -0.5), ("d_R", -1/3, 4.7e-3, 0.0)],
        "ss": [("s_L", -1/3, 0.096, -0.5), ("s_R", -1/3, 0.096, 0.0)],
    }
    for label, states in quark_groups.items():
        mq = states[0][2]  # threshold
        sum_cqq = np.zeros_like(ma_grid, dtype=complex)
        for name, Qf, mf, T3 in states:
            sum_cqq += c_qq(Qf, T3, mf, ma_array=ma_grid)
        
        form_factor = 1 + (83 / 4) * alpha_s_vals / np.pi
        prefactor = 32 * np.pi * alpha_s_vals**2 * ma_grid**3 / Lambda_grid**2
        amplitude = Cgg + sum_cqq / (32 * np.pi**2)
        amplitude_sq = amplitude * np.conj(amplitude)
        
        mask = ma_grid >= 2 * mq
        result = np.zeros_like(ma_grid, dtype=complex)
        result[mask] = prefactor[mask] * form_factor[mask] * amplitude_sq[mask]
        results[label] = np.real(result)
    return results

def Gamma_a_to_QQ(ma_grid, mQ, cQQ_eff, Lambda_grid):
    Gamma = np.zeros_like(ma_grid)
    mask = ma_grid >= 2 * mQ
    m = ma_grid[mask]
    L = Lambda_grid[mask]
    kin = np.sqrt(1 - 4 * mQ**2 / m**2)
    pref = 3 * m * mQ**2 / (8 * np.pi * L**2)
    ce2 = np.abs(cQQ_eff[mask])**2
    Gamma[mask] = pref * ce2 * kin
    return Gamma

def compute_heavy_quark_decay_widths(ma_grid, Lambda_grid):
    results = {}
    quarks = {
        "cc": [("c_L", 2/3, 1.28, +0.5), ("c_R", 2/3, 1.28, 0.0)],
        "bb": [("b_L", -1/3, 4.18, -0.5), ("b_R", -1/3, 4.18, 0.0)],
    }
    for label, quark_states in quarks.items():
        Gamma_sum = np.zeros_like(ma_grid)
        for name, Qf, mf, T3 in quark_states:
            c_eff = c_qq(Qq=Qf, T3q=T3, mq=mf, ma_array=ma_grid)
            Gamma = Gamma_a_to_QQ(ma_grid, mQ=mf, cQQ_eff=c_eff, Lambda_grid=Lambda_grid)
            Gamma_sum += Gamma
        results[label] = Gamma_sum
    return results

In [ ]:
Gamma_gg = Gamma_a_to_gg(ma_grid, Lambda_grid, build_sm_fermions(ma_grid))
Gamma_ll = compute_lepton_decay_widths(ma_grid, Lambda_grid)
Gamma_lqlq = compute_light_quark_decay_widths(ma_grid, Lambda_grid)
Gamma_qq = compute_heavy_quark_decay_widths(ma_grid, Lambda_grid)

In [ ]:
np.shape(Gamma_gg),np.shape(Gamma_ll['e']),np.shape(Gamma_lqlq['uu']),np.shape(Gamma_qq['bb'])

In [ ]:
Gamma_ll_total   = reduce(operator.add, Gamma_ll.values())
Gamma_lqlq_total = reduce(operator.add, Gamma_lqlq.values())
Gamma_qq_total   = reduce(operator.add, Gamma_qq.values())
Gamma_total  = Gamma_gg + Gamma_ll_total + Gamma_lqlq_total + Gamma_qq_total

In [ ]:
Ea = (mz**2 + ma_grid**2) / (2 * mz)
La = hbar_c * Ea / (ma_grid * Gamma_total)
# Decay probability functions
def P_decay(Lmin, Lmax, Ld):
    return np.exp(-Lmin / Ld) - np.exp(-Lmax / Ld)
Prompt = P_decay(0, 1, La)

In [ ]:
# 创建一个DataFrame来保存所有衰变宽度
output_df = pd.DataFrame({
    'ma': ma_grid,
    'Lambda': Lambda_grid,
    'BR_gg': Gamma_gg/Gamma_total,
    'BR_ee': Gamma_ll['e']/Gamma_total,
    'BR_mumu': Gamma_ll['mu']/Gamma_total,
    'BR_tautau': Gamma_ll['tau']/Gamma_total,
    'BR_uu': Gamma_lqlq['uu']/Gamma_total,
    'BR_dd': Gamma_lqlq['dd']/Gamma_total,
    'BR_ss': Gamma_lqlq['ss']/Gamma_total,
    'BR_cc': Gamma_qq['cc']/Gamma_total,
    'BR_bb': Gamma_qq['bb']/Gamma_total,
    'prompt': Prompt,
})

# 写入CSV文件
output_df.to_csv('alp_decay_widths_frac.csv', index=False)

In [ ]:
# 读取数据
df = pd.read_csv('alp_decay_widths_frac.csv')
ma = df['ma']
columns = ['BR_gg', 'BR_ee', 'BR_mumu', 'BR_tautau', 'BR_uu', 'BR_dd', 'BR_ss', 'BR_cc', 'BR_bb']
labels = [
    r'$\gamma\gamma$',
    r'$e^+e^-$',
    r'$\mu^+\mu^-$',
    r'$\tau^+\tau^-$',
    r'$u\bar{u}$',
    r'$d\bar{d}$',
    r'$s\bar{s}$',
    r'$c\bar{c}$',
    r'$b\bar{b}$'
]

fig, ax = plt.subplots()
for col, lab in zip(columns, labels):
    ax.plot(ma, df[col], label=lab)
ax.set_xscale('log')
#ax.set_ylim(10**(-3), 1.05)
ax.set_yscale('log')
ax.set_xlim(5*10**(-5),100)
ax.set_xlabel(R'm$_{a}$ [GeV]', fontsize=16)
ax.tick_params(axis='both', which='major', labelsize=14)
ax.set_ylabel('Branching Ratio', fontsize=16)
ax.legend(ncol=2)  # 设置图例两列
plt.savefig('branching_ratios_ww1_bb1.pdf', bbox_inches='tight')
plt.show()

In [ ]:
df = pd.read_csv('./XS.csv')
df.columns

In [ ]:
ma_grid = df['MQ'].values
Lambda_grid = df['gstar'].values #100000
xs = df['XS'].values#耦合系数要平方，在csv中已经平方了
xs10 = df['XS10'].values
xs100 = df['XS100'].values

In [ ]:
def Gamma_total_ALP(ma_grid, Lambda_grid):
    Gamma_gg = Gamma_a_to_gg(ma_grid, Lambda_grid, build_sm_fermions(ma_grid))
    Gamma_ll = compute_lepton_decay_widths(ma_grid, Lambda_grid)
    Gamma_lqlq = compute_light_quark_decay_widths(ma_grid, Lambda_grid)
    Gamma_qq = compute_heavy_quark_decay_widths(ma_grid, Lambda_grid)
    Gamma_ll_total   = reduce(operator.add, Gamma_ll.values())
    Gamma_lqlq_total = reduce(operator.add, Gamma_lqlq.values())
    Gamma_qq_total   = reduce(operator.add, Gamma_qq.values())
    Gamma_total  = Gamma_gg + Gamma_ll_total + Gamma_lqlq_total + Gamma_qq_total
    return Gamma_total

In [ ]:
Gamma_gg = Gamma_a_to_gg(ma_grid, Lambda_grid, build_sm_fermions(ma_grid))
Gamma_ll = compute_lepton_decay_widths(ma_grid, Lambda_grid)
Gamma_lqlq = compute_light_quark_decay_widths(ma_grid, Lambda_grid)
Gamma_qq = compute_heavy_quark_decay_widths(ma_grid, Lambda_grid)

In [ ]:
Gamma_ll_total   = reduce(operator.add, Gamma_ll.values())
Gamma_lqlq_total = reduce(operator.add, Gamma_lqlq.values())
Gamma_qq_total   = reduce(operator.add, Gamma_qq.values())
Gamma_total  = Gamma_gg + Gamma_ll_total + Gamma_lqlq_total + Gamma_qq_total

In [ ]:
Gamma_total_1000 = Gamma_total_ALP(ma_grid, Lambda_grid/100)
Gamma_total_10000 = Gamma_total_ALP(ma_grid, Lambda_grid/10)
Gamma_total_100000 = Gamma_total_ALP(ma_grid, Lambda_grid/1)

In [ ]:
Ea = (mz**2 + ma_grid**2) / (2 * mz)
La_1000 = hbar_c * Ea / (ma_grid * Gamma_total_1000)
La_10000 = hbar_c * Ea / (ma_grid * Gamma_total_10000)
La_100000 = hbar_c * Ea / (ma_grid * Gamma_total_100000)
# Decay probability functions
def P_decay(Lmin, Lmax, Ld):
    return np.exp(-Lmin / Ld) - np.exp(-Lmax / Ld)
Prompt_1000 = P_decay(0, 1, La_1000)
Prompt_10000 = P_decay(0, 1, La_10000)
Prompt_100000 = P_decay(0, 1, La_100000)

In [ ]:
# 创建一个DataFrame来保存所有衰变宽度
output_df = pd.DataFrame({
    'ma': ma_grid,
    'Lambda': Lambda_grid,
    'xs': xs,
    'xs_prompt': xs * Prompt_1000 ,
    'xs10': xs10,
    'xs10_prompt': xs10 * Prompt_10000 ,
    'xs100': xs100,
    'xs100_prompt': xs100 * Prompt_100000 ,
})

# 写入CSV文件
output_df.to_csv('alp_cross_section_ww0_bb1.csv', index=False)

In [ ]:
# 读取数据
df = pd.read_csv('./alp_cross_section_ww0_bb1.csv')
ma_grid = df['ma']
columns = ['xs', 'xs_prompt', 'xs10', 'xs10_prompt', 'xs100', 'xs100_prompt']
labels = [
    R'$\sigma_{total}$ at $\Lambda$ = 1 TeV' ,
    R'$\sigma_{prompt}$ at $\Lambda$ = 1 TeV' ,
    R'$\sigma_{total}$ at $\Lambda$ = 10 TeV',
    R'$\sigma_{prompt}$ at $\Lambda$ = 10 TeV',
    R'$\sigma_{total}$ at $\Lambda$ = 100 TeV',
    R'$\sigma_{prompt}$ at $\Lambda$ = 100 TeV',
]

# 设置颜色和线型
colors = ['black', 'black', 'red', 'red', 'blue', 'blue']
linestyles = ['-', '--', '-', '--', '-', '--']

fig, ax = plt.subplots()
for col, lab, color, linestyle in zip(columns, labels, colors, linestyles):
    ax.plot(ma_grid, df[col], label=lab, color=color, linestyle=linestyle, linewidth = 2)

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlim(1e-3,100)
ax.set_xlabel(R'm$_{a}$ [GeV]', fontsize=16)
ax.tick_params(axis='both', which='major', labelsize=14)
ax.set_ylabel(R'$\sigma$ [pb]', fontsize=16)
ax.legend(ncol=1, loc='lower right', frameon=False)  # 设置图例两列
plt.savefig('prompt_ww1_bb1.pdf', bbox_inches='tight')
plt.show()
